# ml-agent на примере титаника

Агент получает задание на русском, сам выбирает инструменты и запускает процессы библиотеки outboxml. Рабочая папка агента — папка этого ноутбука: здесь лежат данные `data/titanic.csv`, конфиг моделей `configs/titanic.json` и конфиг AutoML `configs/automl.json`. Исходные конфиги агент не меняет, а копии и результаты сохраняет рядом.

Инструменты агента:
- `fit_models` — обучает модели конфига и возвращает метрики на train и test;
- `check_features` — проверяет фичи для модели отбором outboxml и сохраняет копию конфига с выбранными фичами.

In [ ]:
import sys
from pathlib import Path

from IPython.display import Markdown
from loguru import logger

from ml_agent.chat import OpenAiChat
from ml_agent.checking import FeatureCheck
from ml_agent.fitting import Fitting
from ml_agent.loop import Loop
from ml_agent.outbox import LocalOutbox
from ml_agent.tools import Tools
from ml_agent.workspace import Folder

logger.remove()
logger.add(sys.stderr, level="WARNING")

## Языковая модель

Агент работает с любым сервисом, совместимым с OpenAI Chat Completions (`/v1/chat/completions`): vLLM, VSK AI, Qwen и другими. Укажите адрес до `/v1`, ключ и имя модели.

In [ ]:
llm = OpenAiChat(
    url="https://<host>/v1",
    key="<api-key>",
    model="<model>",
)

## Агент

Агент собирается из языковой модели, инструментов и лимита шагов. Инструкция ниже — то, что модель узнаёт об инструментах.

In [ ]:
root = Path.cwd()
tools = Tools(
    {
        "fit_models": Fitting(Folder(root), LocalOutbox(root)),
        "check_features": FeatureCheck(Folder(root), LocalOutbox(root)),
    },
)
agent = Loop(llm, tools, 12)
Markdown(tools.manual())

## Задание 1. Проверить обучение

In [ ]:
Markdown(
    agent.answer(
        "Обучи модели из конфига configs/titanic.json и покажи метрики на тестовой "
        "выборке. Есть ли признаки переобучения?",
    ),
)

## Задание 2. Проверить фичи

Агент найдёт фичи в конфиге и датасете, запустит отбор только для новых фич и сохранит копию конфига `configs/titanic_first_features.json`, если какие-то фичи будут выбраны.

In [ ]:
Markdown(
    agent.answer(
        "Проверь фичи SEX, PCLASS, EMBARKED, CABIN, NAME и DECK для модели first. "
        "Конфиг моделей — configs/titanic.json, конфиг AutoML — configs/automl.json. "
        "Расскажи про группировки и тип выбранных фич и попали ли они в топ.",
    ),
)

## Задание 3. Обучить модель с выбранными фичами

Каждое задание агент выполняет с нуля, поэтому в промпте указаны оба конфига.

In [ ]:
Markdown(
    agent.answer(
        "Обучи модели из конфигов configs/titanic.json и "
        "configs/titanic_first_features.json и сравни метрики модели first "
        "на тестовой выборке. Стоит ли оставить новые фичи?",
    ),
)